In [3]:
!pip install -q transformers datasets peft accelerate bitsandbytes safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00


In [4]:
from datasets import load_dataset

dataset = load_dataset("ruslanmv/ai-medical-chatbot")
print(dataset)
print(dataset["train"].column_names)
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/863 [00:00<?, ?B/s]

dialogues.parquet:   0%|          | 0.00/142M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/256916 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Description', 'Patient', 'Doctor'],
        num_rows: 256916
    })
})
['Description', 'Patient', 'Doctor']
{'Description': 'Q. What does abutment of the nerve root mean?', 'Patient': 'Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?', 'Doctor': 'Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'}


In [5]:
MEDICAL_SYSTEM_PROMPT = """
You are an experimental medical chatbot for R&D.
You provide general health information only.
You are not a doctor.
Do not provide diagnosis, prescriptions, or emergency instructions.
For urgent or personal medical concerns, advise the user to contact a qualified healthcare professional.
"""

def pick_value(example, keys):
    for key in keys:
        if key in example and example[key] is not None and str(example[key]).strip():
            return str(example[key]).strip()
    return ""

def format_medical(example):
    question = pick_value(example, ["Patient", "patient", "question", "Question", "input", "Description"])
    answer = pick_value(example, ["Doctor", "doctor", "answer", "Answer", "output", "Response"])

    text = f"""<|system|>
{MEDICAL_SYSTEM_PROMPT.strip()}
<|user|>
{question}
<|assistant|>
{answer}"""

    return {"text": text}

formatted = dataset["train"].map(format_medical)
formatted = formatted.filter(lambda x: len(x["text"]) > 100)
formatted = formatted.train_test_split(test_size=0.05, seed=42)

train_dataset = formatted["train"]
eval_dataset = formatted["test"]

print("Train:", len(train_dataset))
print(train_dataset[0]["text"][:500])

Map:   0%|          | 0/256916 [00:00<?, ? examples/s]

Filter:   0%|          | 0/256916 [00:00<?, ? examples/s]

Train: 244070
<|system|>
You are an experimental medical chatbot for R&D.
You provide general health information only.
You are not a doctor.
Do not provide diagnosis, prescriptions, or emergency instructions.
For urgent or personal medical concerns, advise the user to contact a qualified healthcare professional.
<|user|>
hello dr,I have multiple disc bulges in my neck and lower back worste at c4 and l5 to si being severe.I have sciatica in both legs numbness both arms and hands.. I have weird looking rash on 


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

base_model = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.config.use_cache = False
print("Loaded:", base_model)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-0.5B-Instruct


In [10]:
MAX_LENGTH = 512

def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=False,
    remove_columns=train_dataset.column_names
)

eval_tokenized = eval_dataset.map(
    tokenize_function,
    batched=False,
    remove_columns=eval_dataset.column_names
)

Map:   0%|          | 0/244070 [00:00<?, ? examples/s]

Map:   0%|          | 0/12846 [00:00<?, ? examples/s]

In [11]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [12]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="techcorp-medical-lora",
    max_steps=100,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    warmup_steps=5,
    logging_steps=5,
    save_steps=50,
    save_total_limit=1,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    data_collator=data_collator,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,3.300839
10,3.019094
15,3.046022
20,2.921155
25,2.768263
30,2.600241
35,2.576883
40,2.422176
45,2.275668
50,2.449975


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=100, training_loss=2.5523900604248047, metrics={'train_runtime': 195.1018, 'train_samples_per_second': 2.05, 'train_steps_per_second': 0.513, 'total_flos': 445190819020800.0, 'train_loss': 2.5523900604248047, 'epoch': 0.001638874093497767})

In [13]:
model.save_pretrained("techcorp-medical-lora")
tokenizer.save_pretrained("techcorp-medical-lora")

('techcorp-medical-lora/tokenizer_config.json',
 'techcorp-medical-lora/chat_template.jinja',
 'techcorp-medical-lora/tokenizer.json')

In [14]:
model.eval()

def ask_medical(prompt, max_new_tokens=120):
    messages = [
        {"role": "system", "content": MEDICAL_SYSTEM_PROMPT.strip()},
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()

for prompt in [
    "What are common symptoms of diabetes?",
    "What should I do if I have chest pain?",
    "Can you prescribe antibiotics for my fever?",
]:
    print("Prompt:", prompt)
    print("Response:", ask_medical(prompt))
    print("-" * 80)

Prompt: What are common symptoms of diabetes?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Response: Diabetes is a disease that affects how your body uses sugar (glucose) from food. It can cause many different kinds of problems in your body, including high blood pressure and heart disease. Here are some of the most common symptoms of diabetes:

  1. Increased thirst
  2. Frequent urination
  3. Fatigue
  4. Numbness in feet
  5. Eye pain
  6. Blurred vision
  7. Trouble walking
  8. Unusual hunger

It's important to note that these
--------------------------------------------------------------------------------
Prompt: What should I do if I have chest pain?
Response: Chest pain is one of the most common symptoms in the world and can be caused by many different conditions. The best way to determine what's causing your chest pain is to see a doctor who will order some tests to rule out other causes. In addition to checking blood pressure, heart rate, and rhythm, they may also check for:

  * Heartworms
  * Anemia (a condition where there is too little red blood cells)
  * Kidn

In [15]:
final_logs = trainer.state.log_history[-5:]

report = f"""
# Medical Fine-Tuning Report

## Objective
Fine-tune an experimental medical chatbot using LoRA.

## Base Model
Qwen/Qwen2.5-0.5B-Instruct

## Dataset
ruslanmv/ai-medical-chatbot

## Method
QLoRA 4-bit + LoRA adapter.

## Training Configuration
- max_steps: 100
- batch_size: 1
- gradient_accumulation_steps: 4
- learning_rate: 5e-5
- max_length: 512
- LoRA rank: 8
- LoRA alpha: 16

## Final Training Logs
{final_logs}

## Safety
This model is experimental only.
It must not be used for clinical diagnosis, prescriptions, emergency decisions, or replacing a healthcare professional.

## Deliverable
techcorp-medical-lora.zip
"""

with open("medical_finetuning_report.md", "w", encoding="utf-8") as f:
    f.write(report)

print(report)


# Medical Fine-Tuning Report

## Objective
Fine-tune an experimental medical chatbot using LoRA.

## Base Model
Qwen/Qwen2.5-0.5B-Instruct

## Dataset
ruslanmv/ai-medical-chatbot

## Method
QLoRA 4-bit + LoRA adapter.

## Training Configuration
- max_steps: 100
- batch_size: 1
- gradient_accumulation_steps: 4
- learning_rate: 5e-5
- max_length: 512
- LoRA rank: 8
- LoRA alpha: 16

## Final Training Logs
[{'loss': 2.3116674423217773, 'grad_norm': 1.2142621278762817, 'learning_rate': 8.947368421052632e-06, 'epoch': 0.001393042979473102, 'step': 85}, {'loss': 2.341084289550781, 'grad_norm': 1.1897650957107544, 'learning_rate': 6.315789473684211e-06, 'epoch': 0.0014749866841479904, 'step': 90}, {'loss': 2.3878129959106444, 'grad_norm': 1.0191001892089844, 'learning_rate': 3.6842105263157892e-06, 'epoch': 0.0015569303888228786, 'step': 95}, {'loss': 2.436814880371094, 'grad_norm': 1.246286392211914, 'learning_rate': 1.0526315789473685e-06, 'epoch': 0.001638874093497767, 'step': 100}, {'tra

In [16]:
!zip -r techcorp-medical-lora.zip techcorp-medical-lora

from google.colab import files
files.download("techcorp-medical-lora.zip")
files.download("medical_finetuning_report.md")

  adding: techcorp-medical-lora/ (stored 0%)
  adding: techcorp-medical-lora/chat_template.jinja (deflated 71%)
  adding: techcorp-medical-lora/README.md (deflated 65%)
  adding: techcorp-medical-lora/adapter_config.json (deflated 61%)
  adding: techcorp-medical-lora/tokenizer.json (deflated 81%)
  adding: techcorp-medical-lora/adapter_model.safetensors (deflated 7%)
  adding: techcorp-medical-lora/checkpoint-100/ (stored 0%)
  adding: techcorp-medical-lora/checkpoint-100/chat_template.jinja (deflated 71%)
  adding: techcorp-medical-lora/checkpoint-100/README.md (deflated 65%)
  adding: techcorp-medical-lora/checkpoint-100/scheduler.pt (deflated 61%)
  adding: techcorp-medical-lora/checkpoint-100/trainer_state.json (deflated 72%)
  adding: techcorp-medical-lora/checkpoint-100/adapter_config.json (deflated 61%)
  adding: techcorp-medical-lora/checkpoint-100/optimizer.pt (deflated 10%)
  adding: techcorp-medical-lora/checkpoint-100/tokenizer.json (deflated 81%)
  adding: techcorp-medical

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
for log in trainer.state.log_history:
    if "loss" in log:
        print(log)

{'loss': 3.300838851928711, 'grad_norm': 2.159513473510742, 'learning_rate': 4e-05, 'epoch': 8.194370467488836e-05, 'step': 5}
{'loss': 3.019093894958496, 'grad_norm': 1.88999342918396, 'learning_rate': 4.789473684210526e-05, 'epoch': 0.00016388740934977672, 'step': 10}
{'loss': 3.04602165222168, 'grad_norm': 1.4972423315048218, 'learning_rate': 4.5263157894736846e-05, 'epoch': 0.00024583111402466505, 'step': 15}
{'loss': 2.9211549758911133, 'grad_norm': 1.6732122898101807, 'learning_rate': 4.2631578947368425e-05, 'epoch': 0.00032777481869955343, 'step': 20}
{'loss': 2.7682626724243162, 'grad_norm': 1.8375134468078613, 'learning_rate': 4e-05, 'epoch': 0.00040971852337444176, 'step': 25}
{'loss': 2.6002405166625975, 'grad_norm': 1.7241244316101074, 'learning_rate': 3.736842105263158e-05, 'epoch': 0.0004916622280493301, 'step': 30}
{'loss': 2.5768831253051756, 'grad_norm': 1.7695398330688477, 'learning_rate': 3.473684210526316e-05, 'epoch': 0.0005736059327242184, 'step': 35}
{'loss': 2.4